In [1]:
! pip install chembl-webresource-client rdkit pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 1.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.2/34.2 MB 44.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 3.3 MB/s eta 0:00:00


In [5]:
# Import libraries
from chembl_webresource_client.new_client import new_client
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
import pandas as pd
import logging
from tqdm import tqdm
import time
import random
import os
from datetime import datetime
import requests.exceptions

In [6]:
def setup_logging():
    """
    Configure comprehensive logging with both console and file outputs.
    
    Returns:
        logging.Logger: Configured logger object
    """
    # Create logs directory if it doesn't exist
    os.makedirs('logs', exist_ok=True)
    
    # Generate unique log filename with timestamp
    log_filename = f'logs/chembl_data_retrieval_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
    
    # Configure logger
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s: %(message)s',
        handlers=[
            logging.FileHandler(log_filename),
            logging.StreamHandler()
        ]
    )
    
    return logging.getLogger(__name__)

def get_targets(logger, query="estrogen receptor", max_retries=3):
    """
    Retrieve targets from ChEMBL with robust error handling and retry mechanism.
    
    Args:
        logger (logging.Logger): Logging object
        query (str): Search query for targets
        max_retries (int): Maximum number of retry attempts
    
    Returns:
        list: List of target dictionaries
    """
    for attempt in range(max_retries):
        try:
            target_query = new_client.target
            targets = list(target_query.filter(target_synonym__icontains=query))
            
            if not targets:
                logger.warning(f"No targets found for query: {query}")
                return []
            
            logger.info(f"Found {len(targets)} targets matching '{query}'")
            return targets
        
        except requests.exceptions.RequestException as e:
            logger.error(f"Network error on attempt {attempt + 1}: {e}")
            
            if attempt < max_retries - 1:
                wait_time = (attempt + 1) * 2  # Exponential backoff
                logger.info(f"Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                logger.error("Max retries reached. Unable to fetch targets.")
                return []

def get_bioactivity_data(logger, target_chembl_id, max_retries=3):
    """
    Retrieve bioactivity data with comprehensive error handling and data cleaning.
    
    Args:
        logger (logging.Logger): Logging object
        target_chembl_id (str): ChEMBL target identifier
        max_retries (int): Maximum number of retry attempts
    
    Returns:
        pd.DataFrame: Cleaned bioactivity data
    """
    for attempt in range(max_retries):
        try:
            activity_query = new_client.activity
            activities = list(activity_query.filter(
                target_chembl_id=target_chembl_id
            ).only(
                ['molecule_chembl_id', 'canonical_smiles', 
                 'standard_type', 'standard_value', 'standard_units']
            ))
            
            if not activities:
                logger.warning(f"No bioactivity data for target {target_chembl_id}")
                return pd.DataFrame()
            
            # Convert to DataFrame and clean
            activity_data = pd.DataFrame(activities)
            activity_data.dropna(subset=['canonical_smiles'], inplace=True)
            
            logger.info(f"Retrieved {len(activity_data)} bioactivity records")
            return activity_data
        
        except Exception as e:
            logger.error(f"Error retrieving bioactivity data on attempt {attempt + 1}: {e}")
            
            if attempt < max_retries - 1:
                wait_time = random.uniform(1, 5)  # Random wait to prevent rate limiting
                logger.info(f"Waiting {wait_time:.2f} seconds before retry...")
                time.sleep(wait_time)
            else:
                logger.error(f"Failed to retrieve data for target {target_chembl_id}")
                return pd.DataFrame()

def automate_estrogen_receptor_data(
    query="estrogen receptor", 
    output_dir="data", 
    max_targets=None
):
    """
    Comprehensive data retrieval process with progress tracking and error resilience.
    
    Args:
        query (str): Search query for targets
        output_dir (str): Directory to save output files
        max_targets (int, optional): Limit number of targets processed
    
    Returns:
        pd.DataFrame: Combined bioactivity data
    """
    # Setup logging
    logger = setup_logging()
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Retrieve targets
    targets = get_targets(logger, query)
    
    if max_targets:
        targets = targets[:max_targets]
    
    # Prepare progress tracking
    all_data = []
    
    # Use tqdm for progress visualization
    for target in tqdm(targets, desc="Processing Targets", unit="target"):
        target_name = target['pref_name']
        target_chembl_id = target['target_chembl_id']
        
        logger.info(f"Processing Target: {target_name} (ChEMBL ID: {target_chembl_id})")
        
        # Retrieve bioactivity data
        bioactivity_data = get_bioactivity_data(logger, target_chembl_id)
        
        if not bioactivity_data.empty:
            bioactivity_data['Target Name'] = target_name
            bioactivity_data['Target ChEMBL ID'] = target_chembl_id
            all_data.append(bioactivity_data)
        
        # Optional: Small pause to prevent overwhelming the server
        time.sleep(random.uniform(0.5, 2))
    
    # Combine data
    if all_data:
        combined_data = pd.concat(all_data, ignore_index=True)
        
        # Generate unique filename
        output_filename = os.path.join(
            output_dir, 
            f"estrogen_receptor_bioactivity_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        )
        
        # Save data
        combined_data.to_csv(output_filename, index=False)
        logger.info(f"Data saved to {output_filename}")
        
        return combined_data
    else:
        logger.warning("No data retrieved")
        return pd.DataFrame()

def main():
    """
    Main execution function with error handling.
    """
    try:
        combined_data = automate_estrogen_receptor_data(max_targets=50)  # Limit for testing
        
        if not combined_data.empty:
            print(f"Total records retrieved: {len(combined_data)}")
            print(combined_data.head())
        else:
            print("No data retrieved.")
    
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

if __name__ == "__main__":
    main()

Processing Targets: 100%|██████████| 36/36 [42:30<00:00, 70.84s/target]  


Total records retrieved: 37954
                                    canonical_smiles molecule_chembl_id  \
0  Oc1ccc2c(c1)S[C@H](C1CCCC1)[C@H](c1ccc(OCCN3CC...       CHEMBL431611   
1  Oc1ccc2c(c1)S[C@H](C1CCCCCC1)[C@H](c1ccc(OCCN3...       CHEMBL316132   
2  Oc1ccc(-c2oc(-c3ccc(O)cc3)c(-c3ccc(O)cc3)c2-c2...       CHEMBL338926   
3  Oc1ccc(-c2oc(-c3ccc(O)cc3)c(-c3ccc(O)cc3)c2-c2...       CHEMBL338926   
4  CC(C)Cc1c(-c2ccc(O)cc2)nn(-c2ccc(O)cc2)c1-c1cc...       CHEMBL127736   

  standard_type standard_units standard_value  type units value  \
0          IC50             nM            2.5  IC50    nM   2.5   
1          IC50             nM            7.5  IC50    nM   7.5   
2           RBA           None           0.84   RBA  None  0.84   
3           RBA           None            8.7   RBA  None   8.7   
4           RBA           None           75.0   RBA  None  75.0   

               Target Name Target ChEMBL ID  
0  Estrogen receptor alpha        CHEMBL206  
1  Estrogen receptor al

In [7]:
path = "/kaggle/working/data/estrogen_receptor_bioactivity_20250204_012600.csv"

In [8]:
df=pd.read_csv(path)

In [9]:
df.head()

,canonical_smiles,molecule_chembl_id,standard_type,standard_units,standard_value,type,units,value,Target Name,Target ChEMBL ID
0,Oc1ccc2c(c1)S[C@H](C1CCCC1)[C@H](c1ccc(OCCN3CC...,CHEMBL431611,IC50,nM,2.50,IC50,nM,2.50,Estrogen receptor alpha,CHEMBL206
1,Oc1ccc2c(c1)S[C@H](C1CCCCCC1)[C@H](c1ccc(OCCN3...,CHEMBL316132,IC50,nM,7.50,IC50,nM,7.50,Estrogen receptor alpha,CHEMBL206
2,Oc1ccc(-c2oc(-c3ccc(O)cc3)c(-c3ccc(O)cc3)c2-c2...,CHEMBL338926,RBA,NaN,0.84,RBA,NaN,0.84,Estrogen receptor alpha,CHEMBL206
3,Oc1ccc(-c2oc(-c3ccc(O)cc3)c(-c3ccc(O)cc3)c2-c2...,CHEMBL338926,RBA,NaN,8.70,RBA,NaN,8.70,Estrogen receptor alpha,CHEMBL206
4,CC(C)Cc1c(-c2ccc(O)cc2)nn(-c2ccc(O)cc2)c1-c1cc...,CHEMBL127736,RBA,NaN,75.00,RBA,NaN,75.00,Estrogen receptor alpha,CHEMBL206


In [10]:
df["Target Name"].value_counts()

Target Name
Estrogen receptor alpha                                                     18371
Estrogen receptor beta                                                       9574
Estrogen receptor                                                            7897
Estrogen-related receptor gamma                                               652
Estrogen-related receptor alpha                                               573
Estrogen-related receptor beta                                                173
Prohibitin-2                                                                  143
Protein cereblon/Estrogen receptor                                            142
VHL/Cullin-2/Estrogen receptor alpha                                          138
G-protein coupled estrogen receptor 1                                         112
VHL/Estrogen receptor                                                          96
von Hippel-Lindau disease tumor suppressor/Steroid hormone receptor ERR1       46
Prot

In [11]:
df["standard_type"].value_counts()

standard_type
IC50                 10213
Activity              5135
RBA                   4779
Ki                    3051
EC50                  2603
                     ...  
Decrease                 1
Cell number              1
% of control             1
Specific activity        1
Delta Tm                 1
Name: count, Length: 124, dtype: int64